# Demo 2 - Implement a Hybrid Recommender using Content + Collaborative Data

##Scenario: Building a Hybrid Recommender System for Personalized Movie Experiences

CineStream, a growing OTT movie streaming platform, wants to improve user engagement by providing personalized movie recommendations that not only consider user preferences (collaborative filtering) but also incorporate movie content features like genre and director. The company believes this hybrid approach can cater to both popular trends and individual tastes, especially for new or less-rated movies.

##**Objective:**
* Implement a hybrid movie recommender system that combines:

* Collaborative filtering (based on user-movie ratings)

* Content-based filtering (based on movie metadata such as genre and director)

Recommend movies to users that match both their historical preferences and similar content traits.

##Step 1: Load and Prepare the Data

In [ ]:
import pandas as pd

# Load dataset
df = pd.read_csv('hybrid_movie_ratings_demo.csv')

# View the first few rows
print(df.head())


##Step 2: Create User-Movie Rating Matrix (Collaborative Part)

In [ ]:
user_movie_matrix = df.pivot_table(index='user_id', columns='movie_title', values='rating').fillna(0)

##Step 3: Calculate User Similarity (Collaborative Filtering)

In [ ]:
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.metrics.pairwise import cosine_similarity

user_similarity = cosine_similarity(user_movie_matrix)
user_similarity_df = pd.DataFrame(user_similarity, index=user_movie_matrix.index, columns=user_movie_matrix.index)

## Step 4: Prepare Content-Based Features

In [ ]:
# Drop duplicates to avoid repeating movie content
content_df = df[['movie_id', 'movie_title', 'genre', 'director']].drop_duplicates()

# Convert categorical content into numeric using one-hot encoding
content_features = pd.get_dummies(content_df.set_index('movie_title')[['genre', 'director']])

##  Step 5: Compute Content Similarity

In [ ]:
content_similarity = cosine_similarity(content_features)
content_similarity_df = pd.DataFrame(content_similarity, index=content_features.index, columns=content_features.index)

##Step 6: Hybrid Recommendation Function

In [ ]:
def hybrid_recommend(user_id, top_n=5, alpha=0.5):
    # alpha: weight for blending (0=content only, 1=collab only)

    # Movies rated by user
    user_ratings = user_movie_matrix.loc[user_id]
    rated_movies = user_ratings[user_ratings > 0].index.tolist()

    # Content score: weighted sum of similarities from content-based
    content_scores = pd.Series(0, index=user_movie_matrix.columns)
    for movie in rated_movies:
        sim_scores = content_similarity_df[movie]
        content_scores += sim_scores * user_ratings[movie]

    # Collaborative score: weighted average from similar users
    similar_users = user_similarity_df[user_id]
    weighted_ratings = user_movie_matrix.T.dot(similar_users)
    collab_scores = weighted_ratings / similar_users.sum()

    # Combine both
    final_scores = (alpha * collab_scores) + ((1 - alpha) * content_scores)

    # Remove already rated
    final_scores = final_scores.drop(labels=rated_movies, errors='ignore')

    # Top N recommendations
    return final_scores.sort_values(ascending=False).head(top_n)


##Step 7: Run the Hybrid Recommender

In [ ]:
# Example: Recommend movies for user 'U3'
print("Hybrid recommendations for U3:")
print(hybrid_recommend('U3', top_n=5, alpha=0.5))
